# 🚀 AIStudio Colab 外挂多模态车间 (Multimodal Worker v1.0)
## 对应 SOP：AG_SOP_04 全源素材重构与外挂多模态视频流水线

本 Notebook 专为 Google Colab 设计，作为云端 Agent（Antigravity）的外挂多模态 GPU 算力车间：
- 🎙️ **音频转文字与字级时间戳**：`faster-whisper` (Large-v3 / Turbo)
- 🗣️ **零样本声音克隆与旁白合成**：`F5-TTS` / `CosyVoice`
- 🎨 **定制视觉插图与封面渲染**：`FLUX.1-schnell` / `SDXL-Turbo`
- 🎬 **专属动态空镜头与视频生成**：`CogVideoX-2B` / `LTX-Video`
- 📦 **任务总装信箱**：自动挂载 Google Drive，监听并处理 `gap_manifest.json` 差额生产清单

### 步骤 1：挂载 Google Drive 与检查 GPU 环境
请确保在顶部菜单选择 **代码执行程序 -> 更改运行时类型**，选择 **GPU (T4 或 A100)**。

In [ ]:
# 1. 挂载 Google Drive
from google.colab import drive
import os, sys, json, time
drive.mount('/content/drive')

# 2. 检测 GPU 环境
!nvidia-smi
import torch
print(f"\n[GPU Check] PyTorch CUDA 可用性: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[GPU Check] 当前可用 GPU: {torch.cuda.get_device_name(0)}")
    print(f"[GPU Check] 显存容量: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ 警告: 当前未启用 GPU，请切换运行时类型为 GPU！")

### 步骤 2：一键安装多模态核心依赖
安装 `faster-whisper`、`diffusers`、`transformers`、`accelerate` 等生产级依赖。

In [ ]:
!pip install -q --upgrade pip
!pip install -q faster-whisper diffusers transformers accelerate soundfile sentencepiece protobuf
!pip install -q git+https://github.com/SWivid/F5-TTS.git
print("✅ 依赖库安装完成！")

### 步骤 3：定义多模态生产引擎核心类

In [ ]:
import torch
from pathlib import Path

class MultimodalEngine:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.whisper_model = None
        self.image_pipe = None
        self.video_pipe = None
        self.f5_tts_model = None
        print(f"[Engine] 初始化成功，当前主设备: {self.device}")

    def load_whisper(self, model_size="large-v3"):
        if self.whisper_model is None:
            from faster_whisper import WhisperModel
            print(f"[ASR] 加载 faster-whisper ({model_size})...")
            compute_type = "float16" if self.device == "cuda" else "int8"
            self.whisper_model = WhisperModel(model_size, device=self.device, compute_type=compute_type)
        return self.whisper_model

    def transcribe(self, audio_path, output_json_path):
        model = self.load_whisper()
        print(f"[ASR] 开始转写: {audio_path}")
        segments, info = model.transcribe(audio_path, beam_size=5, word_timestamps=True)
        result = {
            "language": info.language,
            "duration": info.duration,
            "segments": []
        }
        for s in segments:
            words = [{"word": w.word, "start": w.start, "end": w.end, "prob": w.probability} for w in (s.words or [])]
            result["segments"].append({
                "id": s.id,
                "start": s.start,
                "end": s.end,
                "text": s.text,
                "words": words
            })
        Path(output_json_path).parent.mkdir(parents=True, exist_ok=True)
        with open(output_json_path, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        print(f"[ASR] 转写完成，输出已保存至: {output_json_path}")
        return result

    def load_image_pipeline(self, model_type="flux-schnell"):
        if self.image_pipe is None:
            from diffusers import AutoPipelineForText2Image
            print(f"[T2I] 加载图像管道 ({model_type})...")
            if model_type == "flux-schnell":
                from diffusers import FluxPipeline
                self.image_pipe = FluxPipeline.from_pretrained(
                    "black-forest-labs/FLUX.1-schnell",
                    torch_dtype=torch.bfloat16
                )
                self.image_pipe.enable_model_cpu_offload()
            else:
                self.image_pipe = AutoPipelineForText2Image.from_pretrained(
                    "stabilityai/sdxl-turbo",
                    torch_dtype=torch.float16,
                    variant="fp16"
                ).to(self.device)
        return self.image_pipe

    def generate_image(self, prompt, output_path, aspect_ratio="16:9", steps=4, seed=None):
        pipe = self.load_image_pipeline()
        width, height = (1024, 576) if aspect_ratio == "16:9" else ((576, 1024) if aspect_ratio == "9:16" else (1024, 1024))
        generator = torch.Generator(device=self.device).manual_seed(seed) if seed is not None else None
        print(f"[T2I] 开始生成图像: {prompt[:40]}...")
        image = pipe(
            prompt=prompt,
            width=width,
            height=height,
            num_inference_steps=steps,
            generator=generator
        ).images[0]
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        image.save(output_path)
        print(f"[T2I] 图像保存成功: {output_path}")
        return output_path

    def generate_tts(self, text, output_path, ref_audio=None, speed=1.0):
        # 使用 F5-TTS 或 Edge-TTS 备用
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        try:
            from f5_tts.api import F5TTS
            if self.f5_tts_model is None:
                print("[TTS] 加载 F5-TTS 模型...")
                self.f5_tts_model = F5TTS()
            print(f"[TTS] F5-TTS 正在合成声音: {text[:30]}...")
            wav, sr, _ = self.f5_tts_model.infer(
                ref_file=ref_audio,
                ref_text="",
                gen_text=text,
                speed=speed,
                file_wave=output_path
            )
            print(f"[TTS] F5-TTS 合成成功: {output_path}")
        except Exception as e:
            print(f"[TTS] F5-TTS 异常 ({e})，切换轻量备选合成...")
            import subprocess
            subprocess.run(["edge-tts", "--text", text, "--write-media", output_path, "--voice", "zh-CN-YunxiNeural"], check=True)
            print(f"[TTS] 备选合成成功: {output_path}")
        return output_path

    def generate_video(self, prompt, output_path, duration_seconds=5, aspect_ratio="16:9"):
        print(f"[T2V] 加载 CogVideoX-2b 视频管道...")
        from diffusers import CogVideoXPipeline
        if self.video_pipe is None:
            self.video_pipe = CogVideoXPipeline.from_pretrained(
                "THUDM/CogVideoX-2b",
                torch_dtype=torch.bfloat16
            )
            self.video_pipe.enable_model_cpu_offload()
            self.video_pipe.vae.enable_slicing()
        print(f"[T2V] 开始生成动态视频片段: {prompt[:40]}...")
        video_frames = self.video_pipe(prompt=prompt, num_frames=duration_seconds*8, guidance_scale=6.0).frames[0]
        from diffusers.utils import export_to_video
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        export_to_video(video_frames, output_path, fps=8)
        print(f"[T2V] 视频生成成功: {output_path}")
        return output_path

engine = MultimodalEngine()
print("✨ MultimodalEngine 实例创建完毕！")

### 步骤 4：定义差额清单调度执行器 (Manifest Runner)
负责解析 `gap_manifest.json`，并自动将产物写回指定的 Google Drive 路径。

In [ ]:
def execute_manifest(manifest_path, drive_project_root="/content/drive/MyDrive/AIStudio"):
    manifest_path = Path(manifest_path)
    if not manifest_path.exists():
        raise FileNotFoundError(f"未找到差额清单: {manifest_path}")

    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)

    project_id = manifest.get("project_id", "default_project")
    tasks = manifest.get("tasks", [])
    style_preset = manifest.get("global_style_preset", {})
    base_dir = Path(drive_project_root) / project_id if drive_project_root else manifest_path.parent

    print(f"\n{'='*50}")
    print(f"📋 开始处理项目任务清单: {project_id}")
    print(f"🎯 总任务数: {len(tasks)}")
    print(f"📁 根目录: {base_dir}")
    print(f"{'='*50}\n")

    for idx, task in enumerate(tasks, start=1):
        task_id = task["task_id"]
        task_type = task["type"]
        params = task.get("params", {})
        out_rel = task.get("output_path")
        out_abs = str(base_dir / out_rel)

        print(f"\n[{idx}/{len(tasks)}] 正在执行: {task_id} (类型: {task_type})")

        if task_type == "tts_narration":
            ref_audio = params.get("ref_audio")
            if ref_audio and not Path(ref_audio).is_absolute():
                ref_audio = str(base_dir / ref_audio)
            engine.generate_tts(
                text=params["text"],
                output_path=out_abs,
                ref_audio=ref_audio,
                speed=params.get("speed", 1.0)
            )
        elif task_type == "text_to_image":
            prompt = params["prompt"]
            if "image_style" in style_preset:
                prompt = f"{prompt}, {style_preset['image_style']}"
            engine.generate_image(
                prompt=prompt,
                output_path=out_abs,
                aspect_ratio=params.get("aspect_ratio", "16:9"),
                steps=params.get("steps", 4),
                seed=params.get("seed", None)
            )
        elif task_type == "text_to_video":
            prompt = params["prompt"]
            if "video_style" in style_preset:
                prompt = f"{prompt}, {style_preset['video_style']}"
            engine.generate_video(
                prompt=prompt,
                output_path=out_abs,
                duration_seconds=params.get("duration_seconds", 5),
                aspect_ratio=params.get("aspect_ratio", "16:9")
            )
        elif task_type == "asr_transcribe":
            audio_file = params["audio_file"]
            if not Path(audio_file).is_absolute():
                audio_file = str(base_dir / audio_file)
            engine.transcribe(audio_file, out_abs)
        else:
            print(f"⚠️ 未知任务类型: {task_type}，跳过")

    # 标记完成
    lock_file = base_dir / "02_generated" / "manifest_completed.lock"
    lock_file.parent.mkdir(parents=True, exist_ok=True)
    with open(lock_file, "w", encoding="utf-8") as f:
        json.dump({"status": "completed", "completed_at": time.time(), "total_tasks": len(tasks)}, f)

    print(f"\n🎉 清单所有任务执行完毕！锁定标记已生成: {lock_file}")

### 步骤 5：启动常驻监听守护循环 (Daemon Mode)
开启此单元格后，Colab 将自动循环监听 Google Drive 中待处理的项目差额清单，发现新清单即刻调动 GPU 跑批。

In [ ]:
def watch_drive_projects(watch_dir="/content/drive/MyDrive/AIStudio/projects", poll_interval=15):
    print(f"👀 开始监听 Google Drive 项目目录: {watch_dir}")
    print(f"⏱️ 轮询间隔: {poll_interval} 秒 (按终止按钮可随时停止)...")
    Path(watch_dir).mkdir(parents=True, exist_ok=True)

    while True:
        try:
            projects = [p for p in Path(watch_dir).iterdir() if p.is_dir()]
            for project in projects:
                manifest_file = project / "04_编排工程" / "gap_manifest.json"
                if not manifest_file.exists():
                    manifest_file = project / "gap_manifest.json"

                lock_file = project / "02_generated" / "manifest_completed.lock"
                if manifest_file.exists() and not lock_file.exists():
                    print(f"\n🔔 发现新任务清单: {manifest_file}")
                    execute_manifest(manifest_file, drive_project_root=str(watch_dir))
            time.sleep(poll_interval)
        except KeyboardInterrupt:
            print("\n🛑 监听已主动停止。")
            break
        except Exception as e:
            print(f"⚠️ 监听异常: {e}")
            time.sleep(poll_interval)

# 若要启动常驻监听，取消下一行注释即可运行：
# watch_drive_projects()